<a href="https://colab.research.google.com/github/JaimRM/QuantitativeFinance/blob/main/Acciona_LBO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Enterprise Value de Acciona
       
       ├──> Equity Purchase Price + Deuda Neta
       │           │
       │           └──> Financiado por: [Deuda LBO (60%)] + [Equity Sponsor (40%)]
       │
       └──> Proyección de EBITDA (6%) ──> [EBITDA - Intereses - Impuestos - CapEx]
                                                        │
                                                        └──> Generación de FCF
                                                                  │
                                                                  └──> Repago de principal de Deuda LBO

In [2]:
import numpy as np
import pandas as pd

# ==========================================
# 1. SUPUESTOS DE ENTRADA (M&A / LBO)
# ==========================================
# Cifras en millones de EUR
año_cero = 2025
precio_accion = 250.0 # Ejemplo ilustrativo
acciones_circulantes = 54.51 # En millones
prima_adquisicion = 0.30 # 30% de prima sobre el precio de mercado

# Valor patrimonial y enterprise value simplificado
equity_purchase_price = acciones_circulantes * precio_accion * (1 + prima_adquisicion)
deuda_neta_asumida = 5500.0 # Deuda neta consolidada estimada
cash_acquired = 1200.0
enterprise_value = equity_purchase_price + deuda_neta_asumida - cash_acquired

# Supuestos operativos y de salida
horizonte_años = 5
tasa_crecimiento_ebitda = 0.06 # Crecimiento anual del EBITDA
margen_ebitda = 0.22 # 22% de margen sobre ventas
multiplicador_ebitda_entrada = 10.5
multiplicador_ebitda_salida = 10.0

# Estructura de capital (Financiación de la adquisición)
porcentaje_deuda_lbo = 0.60
tasa_interes_deuda_lbo = 0.055 # 5.5% de coste de la deuda LBO
tasa_fiscal = 0.25 # 25% impuesto de sociedades

# ==========================================
# 2. PROYECCIÓN OPERATIVA (EBITDA & FCF)
# ==========================================
ebitda_inicial = enterprise_value / multiplicador_ebitda_entrada # Aproximación implícita inicial
ebitda_proyectado = [ebitda_inicial * ( (1 + tasa_crecimiento_ebitda) ** i ) for i in range(1, horizonte_anos + 1)]

# Asumimos que el FCF para repagar deuda es EBITDA - Intereses - Impuestos - CapEx Neto
capex_anual = 800.0 # CapEx de mantenimiento y crecimiento
variacion_wc = -50.0 # Necesidades operativas de capital

print(f"Precio de Compra del Equity (Equity Value): {equity_purchase_price:,.2f} M€")
print(f"Enterprise Value de Entrada: {enterprise_value:,.2f} M€\n")
print("--- Proyección de EBITDA ---")
for i, ebitda in enumerate(ebitda_proyectado, start=1):
    print(f"Año {i} (202{5+i}): {ebitda:,.2f} M€")

# ==========================================
# 3. MECÁNICA DE DEUDA Y REPAGO (DEBT SCHEDULE)
# ==========================================
deuda_inicial = equity_purchase_price * porcentaje_deuda_lbo
deuda_restante = deuda_inicial
cash_flow_disponible = []
calendario_deuda = []

print("\n--- Calendario de Repago de Deuda ---")
for año in range(1, horizonte_años + 1):
    ebitda_año = ebitda_proyectado[año-1]
    intereses_anuales = deuda_restante * tasa_interes_deuda_lbo
    ebt = ebitda_año - intereses_anuales - (capex_anual * 0.1) # Depreciación estimada simplificada
    impuestos = max(0, ebt * tasa_fiscal)

    # FCF disponible para deuda (asumiendo reinversión de CapEx)
    fcf_deuda = ebitda_año - intereses_anuales - impuestos - capex_anual + variacion_wc
    cash_flow_disponible.append(fcf_deuda)

    # Amortización de principal
    principal_pagado = min(fcf_deuda, deuda_restante)
    deuda_restante = max(0.0, deuda_restante - principal_pagado)

    calendario_deuda.append({
        "Año": año,
        "Deuda Inicio": deuda_restante + principal_pagado,
        "Intereses": intereses_anuales,
        "Principal Pagado": principal_pagado,
        "Deuda Fin": deuda_restante
    })

df_deuda = pd.DataFrame(calendario_deuda)
print(df_deuda.to_string(index=False))

# ==========================================
# 4. RENTABILIDAD (RETURNS - IRR Y MOIC)
# ==========================================
ebitda_salida = ebitda_proyectado[-1]
ev_salida = ebitda_salida * multiplicador_ebitda_salida
equity_valor_salida = ev_salida - deuda_restante

inversion_equity_inicial = equity_purchase_price * (1 - porcentaje_deuda_lbo)
moic = equity_valor_salida / inversion_equity_inicial
irr = (moic ** (1 / horizonte_anos)) - 1

print("\n--- Resultados de Salida (Returns) ---")
print(f"Valor del Equity al final del Año {horizonte_anos}: {equity_valor_salida:,.2f} M€")
print(f"Múltiplo sobre el capital invertido (MOIC): {moic:,.2f}x")
print(f"Tasa Interna de Retorno (IRR): {irr*100:.2f}%")

Precio de Compra del Equity (Equity Value): 17,715.75 M€
Enterprise Value de Entrada: 22,015.75 M€

--- Proyección de EBITDA ---
Año 1 (2026): 2,222.54 M€
Año 2 (2027): 2,355.89 M€
Año 3 (2028): 2,497.25 M€
Año 4 (2029): 2,647.08 M€
Año 5 (20210): 2,805.91 M€

--- Calendario de Repago de Deuda ---
 Año  Deuda Inicio  Intereses  Principal Pagado    Deuda Fin
   1  10629.450000 584.619750        398.441973 10231.008027
   2  10231.008027 562.705441        514.892112  9716.115915
   3   9716.115915 534.386375        642.146683  9073.969232
   4   9073.969232 499.068308        781.011421  8292.957811
   5   8292.957811 456.112680        932.346902  7360.610909

--- Resultados de Salida (Returns) ---
Valor del Equity al final del Año 5: 20,698.47 M€
Múltiplo sobre el capital invertido (MOIC): 2.92x
Tasa Interna de Retorno (IRR): 23.91%
